# 02 — Building an Indexing Pipeline That Retrieves Well

Builds the Azure AI Search index for the five driver's manuals. Run `setup/00_setup.ipynb` first.


## Setup — run once per kernel session

In [ ]:
import time

from azure.core.exceptions import HttpResponseError
from azure.identity import DefaultAzureCredential
from azure.search.documents import SearchClient
from azure.search.documents.indexes import SearchIndexClient, SearchIndexerClient
from azure.search.documents.indexes.models import (
    AzureOpenAIEmbeddingSkill,
    AzureOpenAIVectorizer,
    AzureOpenAIVectorizerParameters,
    CognitiveServicesAccountKey,
    FieldMapping,
    FieldMappingFunction,
    HnswAlgorithmConfiguration,
    IndexingParameters,
    IndexingParametersConfiguration,
    IndexProjectionMode,
    InputFieldMappingEntry,
    MergeSkill,
    OcrSkill,
    OutputFieldMappingEntry,
    SearchField,
    SearchFieldDataType,
    SearchIndex,
    SearchIndexer,
    SearchIndexerDataContainer,
    SearchIndexerDataSourceConnection,
    SearchIndexerIndexProjection,
    SearchIndexerIndexProjectionSelector,
    SearchIndexerIndexProjectionsParameters,
    SearchIndexerSkillset,
    SemanticConfiguration,
    SemanticField,
    SemanticPrioritizedFields,
    SemanticSearch,
    SimpleField,
    SplitSkill,
    VectorSearch,
    VectorSearchProfile,
)
from azure.search.documents.models import VectorizableTextQuery

import sys
sys.path.insert(0, "..")

from config import load_config

config = load_config()
credential = DefaultAzureCredential()

index_client = SearchIndexClient(config.search_endpoint, credential)
indexer_client = SearchIndexerClient(config.search_endpoint, credential)
search_client = SearchClient(config.search_endpoint, config.search_index_name, credential)

DATA_SOURCE_NAME = "state-driver-manuals-blob"
SKILLSET_NAME = "state-driver-manuals-skillset"
INDEXER_NAME = "state-driver-manuals-indexer"
INDEX_NAME = config.search_index_name

print("Search endpoint:", config.search_endpoint)
print("Index name:     ", INDEX_NAME)


In [ ]:
def reset_indexer_with_retry(retries=6, delay=5):
    """Retry reset_indexer(): a just-finished run can look in progress for a few seconds."""
    for attempt in range(retries):
        try:
            indexer_client.reset_indexer(INDEXER_NAME)
            return
        except HttpResponseError as e:
            if attempt == retries - 1:
                raise
            print(f"(reset busy, retrying in {delay}s: {e.message})")
            time.sleep(delay)


def run_and_wait(reset=False, poll_seconds=5):
    """Run the indexer and wait for a new result; reset=True clears change tracking first."""
    if reset:
        reset_indexer_with_retry()
        previous_start = None
    else:
        prior = indexer_client.get_indexer_status(INDEXER_NAME).last_result
        previous_start = prior.start_time if prior else None
    indexer_client.run_indexer(INDEXER_NAME)
    print("Indexer running", end="", flush=True)
    # Wait for a result with a new start_time; the previous run can linger briefly.
    # last_result.status shows a live run; status.status only shows the indexer is operational.
    while True:
        status = indexer_client.get_indexer_status(INDEXER_NAME)
        last = status.last_result
        is_new_result = last is not None and last.start_time != previous_start
        if is_new_result and last.status in ("success", "transientFailure", "persistentFailure"):
            print()
            print(f"Status: {last.status} — {last.item_count} item(s), {len(last.errors)} error(s)")
            for err in last.errors[:5]:
                print("  ERROR:", err.error_message)
            return last
        print(".", end="", flush=True)
        time.sleep(poll_seconds)


def build_index_projection(include_vector=False, include_state=False):
    """Build the full projection each time; get_skillset() returns it as a dict, so it cannot be patched."""
    mappings = [
        InputFieldMappingEntry(name="chunk", source="/document/pages/*"),
        InputFieldMappingEntry(name="title", source="/document/title"),
    ]
    if include_vector:
        mappings.append(InputFieldMappingEntry(name="vector", source="/document/pages/*/vector"))
    if include_state:
        mappings.append(InputFieldMappingEntry(name="state", source="/document/state"))
    return SearchIndexerIndexProjection(
        selectors=[
            SearchIndexerIndexProjectionSelector(
                target_index_name=INDEX_NAME,
                parent_key_field_name="parent_id",
                source_context="/document/pages/*",
                mappings=mappings,
            ),
        ],
        parameters=SearchIndexerIndexProjectionsParameters(
            projection_mode=IndexProjectionMode.SKIP_INDEXING_PARENT_DOCUMENTS,
        ),
    )


def show_hits(results, fields=("state", "title", "chunk"), snippet_len=180):
    for i, r in enumerate(results, start=1):
        line = f"{i}. score={r['@search.score']:.3f}"
        if r.get("@search.reranker_score") is not None:
            line += f"  reranker={r['@search.reranker_score']:.3f}"
        print(line)
        for field in fields:
            if field not in r:
                continue
            value = r[field]
            if field == "chunk" and isinstance(value, str):
                value = value.strip().replace("\n", " ")[:snippet_len] + "..."
            print(f"   {field}: {value}")
        print()


## Clip 1 (3 min) — Data Source, Skillset, Indexer, Index

Builds the pipeline: OCR the image-only pages, split the text into ~2000-character chunks, and index each chunk as its own search document.


In [ ]:
data_source = SearchIndexerDataSourceConnection(
    name=DATA_SOURCE_NAME,
    type="azureblob",
    # ResourceId= with no key: the indexer signs in as the search service's managed identity
    # (granted Storage Blob Data Reader by infra/deploy.sh).
    connection_string=(
        f"ResourceId=/subscriptions/{config.subscription_id}/resourceGroups/"
        f"{config.resource_group}/providers/Microsoft.Storage/storageAccounts/"
        f"{config.storage_account_name}/;"
    ),
    container=SearchIndexerDataContainer(name=config.storage_container_name),
)
indexer_client.create_or_update_data_source_connection(data_source)
print(f"Data source '{DATA_SOURCE_NAME}' ready.")


In [ ]:
# Index schema v1: one chunk per search document.
# `chunk_id` is filled by the index projection below.
index_v1 = SearchIndex(
    name=INDEX_NAME,
    fields=[
        # SimpleField() drops analyzer_name, and a projection key needs the keyword analyzer,
        # so use SearchField.
        SearchField(name="chunk_id", type=SearchFieldDataType.String, key=True,
                    analyzer_name="keyword", filterable=True),
        SimpleField(name="parent_id", type=SearchFieldDataType.String,
                    filterable=True),
        SearchField(name="title", type=SearchFieldDataType.String,
                    filterable=True, sortable=False),
        SearchField(name="chunk", type=SearchFieldDataType.String,
                    searchable=True),
    ],
)
index_client.create_or_update_index(index_v1)
print(f"Index '{INDEX_NAME}' created (v1: title + chunk, no vector field yet).")


In [ ]:
ENABLE_OCR = True  # False skips OCR and image extraction

skills = []
split_source = "/document/content"

if ENABLE_OCR:
    skills.append(OcrSkill(
        context="/document/normalized_images/*",
        default_language_code="en",
        should_detect_orientation=True,
        inputs=[InputFieldMappingEntry(name="image", source="/document/normalized_images/*")],
        outputs=[OutputFieldMappingEntry(name="text", target_name="text")],
    ))
    skills.append(MergeSkill(
        context="/document",
        insert_pre_tag=" ",
        insert_post_tag=" ",
        inputs=[
            InputFieldMappingEntry(name="text", source="/document/content"),
            InputFieldMappingEntry(name="itemsToInsert", source="/document/normalized_images/*/text"),
            InputFieldMappingEntry(name="offsets", source="/document/normalized_images/*/contentOffset"),
        ],
        outputs=[OutputFieldMappingEntry(name="mergedText", target_name="merged_text")],
    ))
    split_source = "/document/merged_text"

skills.append(SplitSkill(
    context="/document",
    text_split_mode="pages",
    maximum_page_length=2000,
    page_overlap_length=500,
    inputs=[InputFieldMappingEntry(name="text", source=split_source)],
    outputs=[OutputFieldMappingEntry(name="textItems", target_name="pages")],
))

index_projection = build_index_projection()

skillset = SearchIndexerSkillset(
    name=SKILLSET_NAME,
    description="OCR image-only pages, merge into content, split into page chunks",
    skills=skills,
    index_projection=index_projection,
)
if ENABLE_OCR:
    # The OCR skill's billing attachment is key-based; the free tier caps OCR at 20 documents/day.
    # The embedding skill and vectorizer below are keyless.
    skillset.cognitive_services_account = CognitiveServicesAccountKey(key=config.ai_services_key)

indexer_client.create_or_update_skillset(skillset)
print(f"Skillset '{SKILLSET_NAME}' ready. OCR enabled: {ENABLE_OCR}")


In [ ]:
indexer = SearchIndexer(
    name=INDEXER_NAME,
    data_source_name=DATA_SOURCE_NAME,
    target_index_name=INDEX_NAME,
    skillset_name=SKILLSET_NAME,
    parameters=IndexingParameters(
        configuration=IndexingParametersConfiguration(
            data_to_extract="contentAndMetadata",
            image_action="generateNormalizedImages" if ENABLE_OCR else "none",
            # query_timeout defaults to "00:05:00" and a blob data source rejects it, so set None.
            query_timeout=None,
        ),
    ),
    field_mappings=[
        FieldMapping(source_field_name="metadata_storage_name", target_field_name="title"),
    ],
)
indexer_client.create_or_update_indexer(indexer)
# reset=True forces a full reprocess: an indexer that ran before skips blobs it already saw,
# even if the index is empty.
result = run_and_wait(reset=True)


In [ ]:
results = search_client.search(search_text="FLAMMABLE radioactive placard", top=3, select=["title", "chunk"])
show_hits(results, fields=("title", "chunk"))


## Clip 2 (4 min) — Integrated Vectorization for Text and Image Content

Adds an embedding skill to the skillset and a vectorizer to the index, so no client-side embedding call is needed.


In [ ]:
# Vector search config: HNSW for storing vectors, plus a vectorizer that turns query text into a vector.
vectorizer = AzureOpenAIVectorizer(
    vectorizer_name="openai-vectorizer",
    parameters=AzureOpenAIVectorizerParameters(
        resource_url=config.openai_endpoint,
        deployment_name=config.openai_embedding_deployment,
        model_name=config.openai_embedding_model,
    ),
    # No api_key: keyless, via the search service's managed identity
    # (granted Cognitive Services OpenAI User by infra/deploy.sh).
)

index_v2 = index_client.get_index(INDEX_NAME)
index_v2.fields.append(
    SearchField(
        name="vector",
        type=SearchFieldDataType.Collection(SearchFieldDataType.Single),
        searchable=True,
        # stored=False is only allowed on fields in the original index, so this field keeps the default.
        vector_search_dimensions=1536,
        vector_search_profile_name="vector-profile",
    )
)
index_v2.vector_search = VectorSearch(
    algorithms=[HnswAlgorithmConfiguration(name="hnsw-config")],
    profiles=[VectorSearchProfile(
        name="vector-profile",
        algorithm_configuration_name="hnsw-config",
        vectorizer_name="openai-vectorizer",
    )],
    vectorizers=[vectorizer],
)
index_client.create_or_update_index(index_v2)
print("Index updated: 'vector' field + vectorizer added.")


In [ ]:
embedding_skill = AzureOpenAIEmbeddingSkill(
    context="/document/pages/*",
    resource_url=config.openai_endpoint,
    deployment_name=config.openai_embedding_deployment,
    model_name=config.openai_embedding_model,
    dimensions=1536,
    inputs=[InputFieldMappingEntry(name="text", source="/document/pages/*")],
    outputs=[OutputFieldMappingEntry(name="embedding", target_name="vector")],
)

skillset = indexer_client.get_skillset(SKILLSET_NAME)
skillset.skills = list(skillset.skills) + [embedding_skill]
skillset.index_projection = build_index_projection(include_vector=True)
indexer_client.create_or_update_skillset(skillset)
print("Skillset updated: embedding skill added, projection now maps 'vector' too.")

result = run_and_wait(reset=True)  # reset: force re-embedding of every chunk


In [ ]:
query = "How much space should I leave in front of me?"  # the manuals say "following distance"

results = search_client.search(
    search_text=None,
    vector_queries=[VectorizableTextQuery(text=query, k_nearest_neighbors=5, fields="vector")],
    top=3,
    select=["title", "chunk"],
)
show_hits(results, fields=("title", "chunk"))


## Clip 3 (3 min) — Filters and Semantic Ranking

Adds a filterable `state` field taken from the blob folder name, then a semantic ranker configuration.


In [ ]:
index_v3 = index_client.get_index(INDEX_NAME)
index_v3.fields.append(
    SearchField(name="state", type=SearchFieldDataType.String, filterable=True, facetable=True)
)
index_client.create_or_update_index(index_v3)

indexer = indexer_client.get_indexer(INDEXER_NAME)
indexer.field_mappings = list(indexer.field_mappings) + [
    FieldMapping(
        source_field_name="metadata_storage_path",
        target_field_name="state",
        # metadata_storage_path is the blob URL; token 4 (split on '/') is the state folder.
        # Re-count the position if your storage account or container names change.
        mapping_function=FieldMappingFunction(name="extractTokenAtPosition", parameters={"delimiter": "/", "position": 4}),
    ),
]
indexer_client.create_or_update_indexer(indexer)

skillset = indexer_client.get_skillset(SKILLSET_NAME)
skillset.index_projection = build_index_projection(include_vector=True, include_state=True)
indexer_client.create_or_update_skillset(skillset)

result = run_and_wait(reset=True)


In [ ]:
query = "How close to a fire hydrant can I park?"  # California, Virginia, Florida: 15 ft; New Jersey: 10 ft

print("-- unfiltered: states mixed together --")
show_hits(search_client.search(search_text=query, top=5, select=["state", "title", "chunk"]))

print("-- filter='state eq \'new-jersey\'' --")
# state values are the lowercase, hyphenated blob folder names, e.g. 'new-jersey'.
show_hits(search_client.search(search_text=query, filter="state eq 'new-jersey'", top=5, select=["state", "title", "chunk"]))


In [ ]:
control_query = "How many feet before a turn must I signal?"  # 100 feet in all five manuals: no filter needed

print("-- unfiltered control question: the answer does not vary by state --")
show_hits(search_client.search(search_text=control_query, top=5, select=["state", "title", "chunk"]))


In [ ]:
index_v4 = index_client.get_index(INDEX_NAME)
index_v4.semantic_search = SemanticSearch(
    configurations=[
        SemanticConfiguration(
            name="semantic-config",
            prioritized_fields=SemanticPrioritizedFields(
                title_field=SemanticField(field_name="title"),
                content_fields=[SemanticField(field_name="chunk")],
            ),
        )
    ],
    default_configuration_name="semantic-config",
)
index_client.create_or_update_index(index_v4)
print("Semantic configuration added (query time only; no indexer re-run).")


In [ ]:
fire_hydrant_query = "How close to a fire hydrant can I park?"

results = search_client.search(
    search_text=fire_hydrant_query,
    query_type="semantic",
    semantic_configuration_name="semantic-config",
    top=5,
    select=["state", "title", "chunk"],
)
show_hits(results, fields=("state", "title", "chunk"))
